### Here we will explore the ragas evaluation framework 
using some metrics provided by the lib 
to use them to calculate specific metrics agianst the rag system 

we will store the results in langsmith 

can also store metrics from other sources or frameworks

many eval frameworks provide the same metrics
this doesnt mean they are calculated in the same way 
so understand how your framework caluclates them 

https://docs.ragas.io/en/stable/

https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/

we will look in to:
- context precision 
- context recall 
- relevancy
- faithfullness

these metrics can be calulated in an llm based way or non llm based
we are focusing on non llm methods

we will use id based context precision and recall

see notion notes for this 

context precision = are we retreiving the correct items from the corpus
(how much signal to nouise)

for effieiency

context recall = are we retrieving all of the relevant data in the corpus 
(did we retrieve all of the available signal)

if we cant retrieve all of the relevant data then we cant use it in the context for the llm - so recall is very important 

these are both very important for smaller models
as they are less likely to produce the correct answer given x nouse threshold

response relevancy = how relevant a response is to the users input 
(does the answer have any irrelevant data? were all the points in the question answered)

faithfullness = how factually consistent is the response 
(it checks the context retrieved and answer generated -> is the answer grounded in the context? has anything been hallucinated)

we will focus on precision and recall

so lets build these metrics in

### RAG Evals

In [104]:
import os
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

install ragas and langchain 
`uv add --dev ragas`

use langchain community 
`uv lock`
`uv sync`

### Download example reference data point from langsmith 

we will see what its like to use langsmiths datasets to run evals
wont have to do this manually in future

In [105]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

In [106]:
ls_client = Client()

In [107]:
dataset = ls_client.read_dataset(
    dataset_name="rag-evaluation-dataset"
)

In [108]:
dataset

Dataset(name='rag-evaluation-dataset', description='RAG evaluation dataset', data_type=<DataType.kv: 'kv'>, id=UUID('42c6780b-24c8-4ce0-886e-389b9d83dd42'), created_at=datetime.datetime(2026, 7, 1, 23, 27, 36, 667443, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 7, 1, 23, 27, 36, 667443, tzinfo=TzInfo(0)), example_count=33, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'macOS-26.4-arm64-arm-64bit', 'sdk_version': '0.9.4', 'runtime_version': '3.12.13', 'langchain_version': None, 'py_implementation': 'CPython', 'langchain_core_version': None}})

In [109]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].outputs

{'ground_truth': 'You have a few options. The yellow kids Bluetooth headphones support both wireless and wired use, include a microphone, and are described as suitable for school and online lessons. The Volkano Ninja wired earphones are made for kids and include safe 85dB volume limiting. You also have a 2-pack of wired earbuds with microphone and volume control that work with 3.5mm devices, though those are more general-purpose rather than specifically for kids.',
 'reference_context_ids': ['B09KQP2H7N', 'B0BBF2VC6X', 'B0CH8DRD6K'],
 'reference_descriptions': ["Kids Wireless Headphones, Adjustable Headband, Stereo Sound, 3.5mm Jack, Kids Bluetooth Headphones, Volume Control, Foldable, Build-in Microphone, Over-Ear Headphones for Kids for School Home, Yellow【WIRELESS & WIRED KIDS HEADPHONES】 Stunning kid headphones, built with the latest high quality 5.0 Bluetooth chip, which allows kids wireless headphones to connect fast and stable, also with 3.5mm jack to connect any device. You can

In [110]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].inputs

{'question': 'I need audio gear for a child using a tablet for school. What options do you have?'}

In [111]:
reference_input = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].inputs
reference_output = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].outputs

so we will run one ragas metric on a single example from the dataset aboce

### RAG pipeline

In [112]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    
    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens, 
            "total_tokens": response.usage.total_tokens
        }

    return response.data[0].embedding

def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="amazon-items-collection-01",
        query=query_embedding, 
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scored = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocess_description"])
        similarity_scored.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])


    return {
        "retrieved_context_ids":  retrieved_context_ids, 
        "retrieved_context": retrieved_context,
        "similarity_scored": similarity_scored,
        "retrieved_context_ratings": retrieved_context_ratings
    }

def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}
    """

    return prompt

from urllib3 import response

def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ], 
        reasoning_effort="none"
    )

    current_run = get_current_run_tree()
    if current_run:
        current_run.metadata["usage_metadata"] = {
            "input_tokens": response.usage.prompt_tokens, 
            "output_tokens": response.usage.completion_tokens,
            "total_tokens": response.usage.total_tokens
        }

    return response.choices[0].message.content

def rag_pipeline(question, topk_k=5):

    retrieved_context = retrieve_data(question, k=topk_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "answer": answer, 
        "question": question, 
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [113]:
rag_pipeline("can i get a charger")

{'answer': 'Yes. We have these charger options available:\n\n1) iPhone Charger 6ft 3Pack (Apple MFi Certified Lightning cables) – Fast charging up to 3A and data sync, compatible with many iPhone and iPad models.\n\n2) Compatible with Notebook Charger (Replacement Charger for Notebook, White-60W) – Note: it’s the “second generation” adapter; you should confirm your specific Mac notebook model before buying.\n\n3) USB C to USB C Cable (INIU 6.6ft, 100W PD 5A) – Fast charging for USB-C devices like Samsung phones, iPad Pro, MacBook Pro, etc.',
 'question': 'can i get a charger',
 'retrieved_context_ids': ['B0BBVJJRHD',
  'B0BGH3H1WM',
  'B0BN1CMWCP',
  'B0C9QZS95R',
  'B0BM9THPDQ'],
 'retrieved_context': ['iPhone Charger 6ft 3Pack, Apple MFi Certified Lightning Cable Fast Charging Long iPhone Charger Cord High Speed Data Sync Cable Compatible iPhone 14 13 12 11 Pro Max XS XR X 8 7 6S 6 Plus SE 5S, iPad【iPhone Cable Fast Charging】:The 8-pin connector with lightning end ensures safe chargi

`uv pip install --reinstall --no-cache openai`

 we can use the metadata above to calculate faithfullness of response and precision of recall

### RAGAS Metrics

lets run some evals agains this pipeline

here we will import two metrics

https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/nvidia_metrics/#example-with-singleturnsample_2

In [131]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import ResponseGroundedness, IDBasedContextPrecision, Faithfulness, ResponseRelevancy, IDBasedContextRecall



/var/folders/n4/kbmbdjz508l95b2dtf8jlt7h0000gn/T/ipykernel_13483/1469740796.py:2: DeprecationWarning: Importing ResponseGroundedness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseGroundedness
  from ragas.metrics import ResponseGroundedness, IDBasedContextPrecision, Faithfulness, ResponseRelevancy, IDBasedContextRecall
/var/folders/n4/kbmbdjz508l95b2dtf8jlt7h0000gn/T/ipykernel_13483/1469740796.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import ResponseGroundedness, IDBasedContextPrecision, Faithfulness, ResponseRelevancy, IDBasedContextRecall
/var/folders/n4/kbmbdjz508l95b2dtf8jlt7h0000gn/T/ipykernel_13483/1469740796.py:2: DeprecationWarning: Importin

Ragas uses langchain wrappers around the models to calculate llm based metrics or embedding capabilities

In [116]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model='gpt-5.4-mini'))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

/var/folders/n4/kbmbdjz508l95b2dtf8jlt7h0000gn/T/ipykernel_13483/3519451575.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model='gpt-5.4-mini'))
/var/folders/n4/kbmbdjz508l95b2dtf8jlt7h0000gn/T/ipykernel_13483/3519451575.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


some of the ragas metrics are llm based so we use the above models to calulcate them

reference input is just a dictionary - with the quesion 

`{'question': 'I need audio gear for a child using a tablet for school. What options do you have?'}`

In [117]:
reference_input

{'question': 'I need audio gear for a child using a tablet for school. What options do you have?'}

reference output is a dictionary with the ground truth i.e context ids, reference_description and ground truth 

In [118]:
reference_output

{'ground_truth': 'You have a few options. The yellow kids Bluetooth headphones support both wireless and wired use, include a microphone, and are described as suitable for school and online lessons. The Volkano Ninja wired earphones are made for kids and include safe 85dB volume limiting. You also have a 2-pack of wired earbuds with microphone and volume control that work with 3.5mm devices, though those are more general-purpose rather than specifically for kids.',
 'reference_context_ids': ['B09KQP2H7N', 'B0BBF2VC6X', 'B0CH8DRD6K'],
 'reference_descriptions': ["Kids Wireless Headphones, Adjustable Headband, Stereo Sound, 3.5mm Jack, Kids Bluetooth Headphones, Volume Control, Foldable, Build-in Microphone, Over-Ear Headphones for Kids for School Home, Yellow【WIRELESS & WIRED KIDS HEADPHONES】 Stunning kid headphones, built with the latest high quality 5.0 Bluetooth chip, which allows kids wireless headphones to connect fast and stable, also with 3.5mm jack to connect any device. You can

In [119]:
result = rag_pipeline(reference_input["question"])

In [120]:
result 

{'answer': 'For a child using a tablet for school, the available options are:\n\n1) Volkano Ninja Kids Earphones (wired, 3.5mm stereo jack) – Red/Blue\n- Wired earbuds with a 3.5mm jack for school/tablet use\n- Designed for hearing protection with a limited sound intensity of 85dB\n- Noise-isolating effect to reduce external ambient sounds\n- Includes a kid-friendly carry case with zip closure, earbud caps (S/M/L), and an earphone clip; also has a built-in Pause/Play button and mic\n\n2) Kids Wireless Headphones (over-ear, Bluetooth + 3.5mm jack) – Yellow\n- Connects wirelessly via Bluetooth 5.0 and also supports a 3.5mm jack as a backup/wired option\n- Over-ear, with adjustable headband and soft ear cushions\n- Built-in microphone for calls/video chats/online lessons\n- Foldable design for travel/storage; includes stereo sound and volume control\n\n3) 2 Pack iPhone Wired Headphones with 3.5mm jack + mic/volume control (Apple MFi certified) \n- Wired 3.5mm earbuds with microphone and r

so we compare this result to the reference result
compare context retrieved, item ids recovered and descriptions returned

now we can calculate the metrics

In [128]:
async def ragas_context_precision_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]

    )

    scorer = IDBasedContextPrecision()

    return await scorer.single_turn_ascore(sample)

In [129]:
await ragas_context_precision_id_based(result, reference_output)

0.6

 the score is 0.6 - as we are rerurning all three from the reference (signal)
 BUT with two extra references (noise)
 3/5 = 0.6

In [133]:
async def ragas_context_recall_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]

    )

    scorer = IDBasedContextRecall()

    return await scorer.single_turn_ascore(sample)

In [134]:
await ragas_context_recall_id_based(result, reference_output)

1.0

as expected this is 1 - as we retrieved all three ids from the reference corpus

faithfullness = how faithfull the answer is to the retrieved conetext 

In [ ]:
async def ragas_faithfullness(run):

    sample = SingleTurnSample(
        user_input=run["question"], 
        response=run["answer"], 
        retrieved_contexts=run["retrieved_context"]
    )

    scorer = Faithfulness(llm=ragas_llm)

    return await scorer.single_turn_ascore(sample)

In [138]:
await ragas_faithfullness(result, reference_output)

0.875

multiple llm runs happening under the hood for this 
so takes a bit longer 
this also doesnt require reference data set
note: theres a cost to this!

the higher the score the better

Response Relevancy next 


In [154]:
async def ragas_relevancy(run, example):

    sample = SingleTurnSample(
        user_input=run["question"], 
        response=run["answer"], 
        retrieved_contexts=run["retrieved_context"]
    )

    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.single_turn_ascore(sample)

In [155]:
result

{'answer': 'For a child using a tablet for school, the available options are:\n\n1) Volkano Ninja Kids Earphones (wired, 3.5mm stereo jack) – Red/Blue\n- Wired earbuds with a 3.5mm jack for school/tablet use\n- Designed for hearing protection with a limited sound intensity of 85dB\n- Noise-isolating effect to reduce external ambient sounds\n- Includes a kid-friendly carry case with zip closure, earbud caps (S/M/L), and an earphone clip; also has a built-in Pause/Play button and mic\n\n2) Kids Wireless Headphones (over-ear, Bluetooth + 3.5mm jack) – Yellow\n- Connects wirelessly via Bluetooth 5.0 and also supports a 3.5mm jack as a backup/wired option\n- Over-ear, with adjustable headband and soft ear cushions\n- Built-in microphone for calls/video chats/online lessons\n- Foldable design for travel/storage; includes stereo sound and volume control\n\n3) 2 Pack iPhone Wired Headphones with 3.5mm jack + mic/volume control (Apple MFi certified) \n- Wired 3.5mm earbuds with microphone and r

In [156]:
reference_output

{'ground_truth': 'You have a few options. The yellow kids Bluetooth headphones support both wireless and wired use, include a microphone, and are described as suitable for school and online lessons. The Volkano Ninja wired earphones are made for kids and include safe 85dB volume limiting. You also have a 2-pack of wired earbuds with microphone and volume control that work with 3.5mm devices, though those are more general-purpose rather than specifically for kids.',
 'reference_context_ids': ['B09KQP2H7N', 'B0BBF2VC6X', 'B0CH8DRD6K'],
 'reference_descriptions': ["Kids Wireless Headphones, Adjustable Headband, Stereo Sound, 3.5mm Jack, Kids Bluetooth Headphones, Volume Control, Foldable, Build-in Microphone, Over-Ear Headphones for Kids for School Home, Yellow【WIRELESS & WIRED KIDS HEADPHONES】 Stunning kid headphones, built with the latest high quality 5.0 Bluetooth chip, which allows kids wireless headphones to connect fast and stable, also with 3.5mm jack to connect any device. You can

In [158]:
score = await ragas_relevancy(result, reference_output) 

In [159]:
print(score)

0.7348806758420704


the higher this score the better

Now lets add evals to these metrics functions
for a single run or for multiple experiments 
to see how changes to our retrievel run, prompt, etc affect the output 

to do this we will create a script that can be run from the terminal to run th evals and analyse them in langsmith - we add this in the /api/evals

we do this next to the code - so we can import the code from the source folder i.e the exact production code